# ANALYSE DES STOCKS.

## I. Importation des donnees nettoyees.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [3]:
dossier_nettoye = "DATA_NETTOYEE"

os.makedirs(dossier_nettoye, exist_ok=True)
fichiers = os.listdir(dossier_nettoye)

print(fichiers)

['.gitkeep', 'achats_nettoye.csv', 'clients_nettoye.csv', 'commerciaux_nettoye.csv', 'depenses_nettoye.csv', 'fournisseurs_nettoye.csv', 'livraisons_nettoye.csv', 'magasins_nettoye.csv', 'produits_nettoye.csv', 'stocks_nettoye.csv', 'ventes_nettoye.csv']


In [4]:
ventes = pd.read_csv(os.path.join(dossier_nettoye, "ventes_nettoye.csv"))
clients = pd.read_csv(os.path.join(dossier_nettoye, "clients_nettoye.csv"))
achats = pd.read_csv(os.path.join(dossier_nettoye, "achats_nettoye.csv"))
produits = pd.read_csv(os.path.join(dossier_nettoye, "produits_nettoye.csv"))
commerciaux = pd.read_csv(os.path.join(dossier_nettoye, "commerciaux_nettoye.csv"))
fournisseurs = pd.read_csv(os.path.join(dossier_nettoye, "fournisseurs_nettoye.csv"))
magasins = pd.read_csv(os.path.join(dossier_nettoye, "magasins_nettoye.csv"))
depenses = pd.read_csv(os.path.join(dossier_nettoye, "depenses_nettoye.csv"))
livraisons = pd.read_csv(os.path.join(dossier_nettoye, "livraisons_nettoye.csv"))
stocks = pd.read_csv(os.path.join(dossier_nettoye, "stocks_nettoye.csv"))

## II. Analyse.

### 1. Produits a faible rotation (reste longtemps en stock).

In [10]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: '%.2f' % x)

# =====================================================================
# ÉTAPE 1 : Calculer les quantités entrées, sorties et le stock disponible
# =====================================================================
# On copie la table 'stocks' et on normalise le type de mouvement en minuscules
mvt = stocks.copy()
mvt['type_mvt'] = mvt['type_mouvement'].astype(str).str.lower().str.strip()

# Quantité totale entrée par produit (valeur absolue pour éviter les signes négatifs)
entrees = (
    mvt[mvt['type_mvt'].str.contains('entr', na=False)]
    .groupby('id_produit')['quantite']
    .sum()
    .abs()
    .reset_index()
    .rename(columns={'quantite': 'quantite_entre_total'})
)

# Quantité totale sortie par produit (valeur absolue pour faciliter la lecture)
sorties = (
    mvt[mvt['type_mvt'].str.contains('sort|vent', na=False)]
    .groupby('id_produit')['quantite']
    .sum()
    .abs()
    .reset_index()
    .rename(columns={'quantite': 'quantite_sortie_total'})
)

# Quantité totale actuellement en stock par produit
stock_dispo = (
    mvt.groupby('id_produit')['quantite']
    .sum()
    .reset_index()
    .rename(columns={'quantite': 'quantite_total_en_stock'})
)

# =====================================================================
# ÉTAPE 2 : Calculer les ventes annuelles moyennes sur les ventes validées
# =====================================================================
# On filtre uniquement les ventes validées et on détermine la durée couverte en années
ventes_valides = ventes[
    ventes['statut_vente'].str.lower().str.contains('valid', na=False)
].copy()
ventes_valides['date_vente'] = pd.to_datetime(ventes_valides['date_vente'])

nb_jours_periode = (
    ventes_valides['date_vente'].max() - ventes_valides['date_vente'].min()
).days
nb_annees = max(nb_jours_periode / 365.25, 1)

# Ventes annuelles moyennes par produit
qte_vendue = (
    ventes_valides.groupby('id_produit')['quantite']
    .sum()
    .reset_index()
    .rename(columns={'quantite': 'qte_vendue_totale'})
)
qte_vendue['ventes_annuelles_moy'] = qte_vendue['qte_vendue_totale'] / nb_annees

# =====================================================================
# ÉTAPE 3 : Fusionner les tables et calculer rotation, valeur et couverture
# =====================================================================
df_rot = (
    produits[['id_produit', 'nom_produit', 'cout_achat_unitaire']]
    .merge(entrees, on='id_produit', how='left')
    .merge(sorties, on='id_produit', how='left')
    .merge(stock_dispo, on='id_produit', how='left')
    .merge(
        qte_vendue[['id_produit', 'ventes_annuelles_moy']],
        on='id_produit',
        how='left',
    )
    .fillna(0)
)

# Valeur du stock = quantité totale en stock * coût d'achat unitaire (si stock > 0)
df_rot['valeur du stock'] = np.where(
    df_rot['quantite_total_en_stock'] > 0,
    df_rot['quantite_total_en_stock'] * df_rot['cout_achat_unitaire'],
    0,
)

# Rotation = ventes annuelles moyennes / quantité totale en stock (si stock > 0)
df_rot['rotation'] = np.where(
    df_rot['quantite_total_en_stock'] > 0,
    df_rot['ventes_annuelles_moy'] / df_rot['quantite_total_en_stock'],
    np.nan,
)

# Couverture_jour = 365 / rotation (si rotation > 0)
df_rot['Couverture_jour'] = np.where(
    df_rot['rotation'] > 0, 365 / df_rot['rotation'], np.nan
)

# =====================================================================
# ÉTAPE 4 : Extraire le Flop 10 (plus faible rotation / plus longue couverture)
# =====================================================================
cols_flop10 = [
    'nom_produit',
    'quantite_entre_total',
    'quantite_sortie_total',
    'rotation',
    'quantite_total_en_stock',
    'valeur du stock',
    'Couverture_jour',
]

flop10_rotation = (
    df_rot[df_rot['quantite_total_en_stock'] > 0]
    .sort_values(by=['Couverture_jour', 'rotation'], ascending=[False, True])
    .head(10)
    .reset_index(drop=True)
)

display(flop10_rotation[cols_flop10])

,nom_produit,quantite_entre_total,quantite_sortie_total,rotation,quantite_total_en_stock,valeur du stock,Couverture_jour
0,Eau Eco 357,2486.00,895.00,0.03,2004.00,2627244.00,12239.53
1,Eau Premium 84,2231.00,1317.00,0.04,2211.00,1397352.00,8985.79
2,Savon Plus 105,1743.00,830.00,0.04,1460.00,21593400.00,8917.02
3,Détergent Plus 18,2333.00,1286.00,0.05,1447.00,6395740.00,7989.57
4,Jus Classique 423,1659.00,1119.00,0.05,1350.00,22628700.00,7607.68
5,Farine Plus 68,1668.00,1127.00,0.05,1317.00,18477510.00,7577.96
6,Sucre Nature 314,1915.00,1014.00,0.05,1638.00,9688770.00,7430.48
7,Farine Plus 315,1129.00,687.00,0.05,981.00,7679268.00,7055.79
8,Détergent Plus 12,1245.00,485.00,0.05,1080.00,13828320.00,6945.36
9,Eau Classique 485,1780.00,1060.00,0.05,1141.00,518014.00,6891.71



•	Interprétation : Ces dix produits affichent une rotation très faible (0,03 à 0,05) et une couverture élevée (6 891,71 à 12 239,53 jours), avec des sorties inférieures aux entrées et une forte immobilisation financière sur Jus Classique 423 (22,63 M FCFA), Savon Plus 105 (21,59 M FCFA) et Farine Plus 68 (18,48 M FCFA).

•	Conclusion métier : Suspendre le réapprovisionnement de ces références, afin de ne pas aggraver l'immobilisation de trésorerie ni l'encombrement des entrepôts.


### 2. Produits a forte rotation (mouvement de sortie plus eleve).

In [11]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: '%.2f' % x)

# =====================================================================
# ÉTAPE 1 : Calculer les quantités entrées, sorties et le stock disponible
# =====================================================================
# On copie la table 'stocks' et on normalise le type de mouvement en minuscules
mvt = stocks.copy()
mvt['type_mvt'] = mvt['type_mouvement'].astype(str).str.lower().str.strip()

# Quantité totale entrée par produit (valeur absolue)
entrees = (
    mvt[mvt['type_mvt'].str.contains('entr', na=False)]
    .groupby('id_produit')['quantite']
    .sum()
    .abs()
    .reset_index()
    .rename(columns={'quantite': 'quantite_entre_total'})
)

# Quantité totale sortie par produit (valeur absolue)
sorties = (
    mvt[mvt['type_mvt'].str.contains('sort|vent', na=False)]
    .groupby('id_produit')['quantite']
    .sum()
    .abs()
    .reset_index()
    .rename(columns={'quantite': 'quantite_sortie_total'})
)

# Quantité totale actuellement en stock par produit
stock_dispo = (
    mvt.groupby('id_produit')['quantite']
    .sum()
    .reset_index()
    .rename(columns={'quantite': 'quantite_total_en_stock'})
)

# =====================================================================
# ÉTAPE 2 : Calculer les ventes annuelles moyennes sur les ventes validées
# =====================================================================
# On garde uniquement les ventes validées et on calcule la durée de la période en années
ventes_valides = ventes[
    ventes['statut_vente'].str.lower().str.contains('valid', na=False)
].copy()
ventes_valides['date_vente'] = pd.to_datetime(ventes_valides['date_vente'])

nb_jours_periode = (
    ventes_valides['date_vente'].max() - ventes_valides['date_vente'].min()
).days
nb_annees = max(nb_jours_periode / 365.25, 1)

# Ventes annuelles moyennes par produit
qte_vendue = (
    ventes_valides.groupby('id_produit')['quantite']
    .sum()
    .reset_index()
    .rename(columns={'quantite': 'qte_vendue_totale'})
)
qte_vendue['ventes_annuelles_moy'] = qte_vendue['qte_vendue_totale'] / nb_annees

# =====================================================================
# ÉTAPE 3 : Fusionner les tables et calculer rotation, valeur et couverture
# =====================================================================
df_rot = (
    produits[['id_produit', 'nom_produit', 'cout_achat_unitaire']]
    .merge(entrees, on='id_produit', how='left')
    .merge(sorties, on='id_produit', how='left')
    .merge(stock_dispo, on='id_produit', how='left')
    .merge(
        qte_vendue[['id_produit', 'ventes_annuelles_moy']],
        on='id_produit',
        how='left',
    )
    .fillna(0)
)

# Valeur du stock = quantité totale en stock * coût d'achat unitaire (si stock > 0)
df_rot['valeur du stock'] = np.where(
    df_rot['quantite_total_en_stock'] > 0,
    df_rot['quantite_total_en_stock'] * df_rot['cout_achat_unitaire'],
    0,
)

# Rotation = ventes annuelles moyennes / quantité totale en stock (si stock > 0)
df_rot['rotation'] = np.where(
    df_rot['quantite_total_en_stock'] > 0,
    df_rot['ventes_annuelles_moy'] / df_rot['quantite_total_en_stock'],
    np.nan,
)

# Couverture_jour = 365 / rotation (si rotation > 0)
df_rot['Couverture_jour'] = np.where(
    df_rot['rotation'] > 0, 365 / df_rot['rotation'], np.nan
)

# =====================================================================
# ÉTAPE 4 : Extraire le Top 10 (plus forte rotation / plus courte couverture)
# =====================================================================
cols_top10 = [
    'nom_produit',
    'quantite_entre_total',
    'quantite_sortie_total',
    'rotation',
    'quantite_total_en_stock',
    'valeur du stock',
    'Couverture_jour',
]

top10_rotation = (
    df_rot[
        (df_rot['quantite_total_en_stock'] > 0) & (df_rot['rotation'] > 0)
    ]
    .sort_values(by=['rotation', 'Couverture_jour'], ascending=[False, True])
    .head(10)
    .reset_index(drop=True)
)

display(top10_rotation[cols_top10])

,nom_produit,quantite_entre_total,quantite_sortie_total,rotation,quantite_total_en_stock,valeur du stock,Couverture_jour
0,Détergent Classique 124,1030.00,1514.00,94.48,1.00,23365.00,3.86
1,Pâtes Plus 249,1201.00,1557.00,40.90,2.00,20210.00,8.92
2,Lait Nature 69,680.00,1153.00,40.06,2.00,32426.00,9.11
3,Café Classique 427,1125.00,1340.00,5.20,17.00,200991.00,70.13
4,Huile Premium 338,1119.00,1528.00,4.92,15.00,268395.00,74.20
5,Huile Classique 361,707.00,1396.00,3.42,20.00,63000.00,106.66
6,Lait Classique 75,1374.00,1857.00,2.35,32.00,743040.00,155.48
7,Détergent Premium 210,1274.00,1787.00,2.32,22.00,52074.00,157.20
8,Café Classique 450,911.00,1366.00,2.23,30.00,415950.00,163.99
9,Biscuit Plus 447,1006.00,1487.00,1.80,45.00,988200.00,203.29


•	Interprétation : Ces dix produits affichent une forte rotation (1,80 à 94,48) avec des sorties supérieures aux entrées et un stock disponible très réduit sur les trois premiers articles : Détergent Classique 124 (1 unité, 3,86 jours), Pâtes Plus 249 (2 unités, 8,92 jours) et Lait Nature 69 (2 unités, 9,11 jours).

•	Conclusion métier : Doubler à tripler les quantités commandées sur ces références à forte rotation et déclencher un réapprovisionnement rapide afin d'éviter toute rupture de stock.


### 3.Produits en rupture.

In [13]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: '%.2f' % x)

# =====================================================================
# ÉTAPE 1 : Calculer la quantité entrée et le stock disponible
# =====================================================================
# On copie la table 'stocks' et on harmonise le type de mouvement en minuscules
mvt = stocks.copy()
mvt['type_mvt'] = mvt['type_mouvement'].astype(str).str.lower().str.strip()

# Quantité totale entrée par produit (valeur absolue)
entrees = (
    mvt[mvt['type_mvt'].str.contains('entr', na=False)]
    .groupby('id_produit')['quantite']
    .sum()
    .abs()
    .reset_index()
    .rename(columns={'quantite': 'quantite_entrer'})
)

# Stock disponible net par produit
stock_dispo = (
    mvt.groupby('id_produit')['quantite']
    .sum()
    .reset_index()
    .rename(columns={'quantite': 'stock_disponible'})
)

# =====================================================================
# ÉTAPE 2 : Calculer les ventes annuelles moyennes sur les ventes validées
# =====================================================================
# On conserve uniquement les ventes validées et on détermine la durée en années
ventes_valides = ventes[
    ventes['statut_vente'].str.lower().str.contains('valid', na=False)
].copy()
ventes_valides['date_vente'] = pd.to_datetime(ventes_valides['date_vente'])

nb_jours_periode = (
    ventes_valides['date_vente'].max() - ventes_valides['date_vente'].min()
).days
nb_annees = max(nb_jours_periode / 365.25, 1)

qte_vendue = (
    ventes_valides.groupby('id_produit')['quantite']
    .sum()
    .reset_index()
    .rename(columns={'quantite': 'qte_vendue_totale'})
)
qte_vendue['ventes_annuelles_moy'] = qte_vendue['qte_vendue_totale'] / nb_annees

# =====================================================================
# ÉTAPE 3 : Fusionner, isoler les ruptures et calculer la rotation
# =====================================================================
df_rupture = (
    produits[['id_produit', 'nom_produit']]
    .merge(entrees, on='id_produit', how='left')
    .merge(stock_dispo, on='id_produit', how='left')
    .merge(
        qte_vendue[['id_produit', 'ventes_annuelles_moy']],
        on='id_produit',
        how='left',
    )
    .fillna(0)
)

# On filtre uniquement les produits en rupture (stock disponible <= 0)
df_rupture = df_rupture[df_rupture['stock_disponible'] <= 0].copy()

# Calcul de la rotation : ventes annuelles / stock si > 0, sinon ventes annuelles moyennes (ou 0)
df_rupture['rotation'] = np.where(
    df_rupture['stock_disponible'] > 0,
    df_rupture['ventes_annuelles_moy'] / df_rupture['stock_disponible'],
    np.where(
        df_rupture['ventes_annuelles_moy'] > 0,
        df_rupture['ventes_annuelles_moy'],
        0.0,
    ),
)

# =====================================================================
# ÉTAPE 4 : Afficher le nombre total de ruptures et le Top 10 trié
# =====================================================================
nb_total_ruptures = len(df_rupture)
print(f'Nombre total de produits en rupture : {nb_total_ruptures}')

top10_rupture = (
    df_rupture.sort_values(by='rotation', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(top10_rupture[['nom_produit', 'quantite_entrer', 'rotation']])

Nombre total de produits en rupture : 197


,nom_produit,quantite_entrer,rotation
0,Détergent Plus 122,1030.00,116.52
1,Détergent Classique 355,1371.00,102.50
2,Jus Premium 4,767.00,101.50
3,Biscuit Plus 284,1310.00,100.16
4,Savon Famille 496,1268.00,99.83
5,Riz Eco 181,900.00,99.83
6,Jus Famille 71,780.00,99.16
7,Biscuit Plus 235,906.00,99.16
8,Café Nature 439,1307.00,98.49
9,Sucre Nature 48,998.00,97.82


•	Interprétation : Sur un total de 197 produits en rupture, ces dix références cumulent des volumes d'entrées élevés (767,00 à 1 371,00 unités) et une rotation très rapide (97,82 à 116,52), traduisant une forte demande non couverte par le stock actuel.

•	Conclusion métier : Prioriser ces références à rotation rapide dans le réapprovisionnement en doublant à triplant les quantités commandées afin de limiter les pertes de ventes.


### 4.Produits en surstock.

In [14]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: '%.2f' % x)

# =====================================================================
# ÉTAPE 1 : Calculer le stock disponible par produit
# =====================================================================
# On additionne les mouvements de 'stocks' par produit pour obtenir le stock disponible
stock_dispo = (
    stocks.groupby('id_produit')['quantite']
    .sum()
    .reset_index()
    .rename(columns={'quantite': 'quantite en stock'})
)

# =====================================================================
# ÉTAPE 2 : Calculer les ventes annuelles moyennes sur les ventes validées
# =====================================================================
# On conserve uniquement les ventes validées pour mesurer le rythme moyen annuel
ventes_valides = ventes[
    ventes['statut_vente'].str.lower().str.contains('valid', na=False)
].copy()
ventes_valides['date_vente'] = pd.to_datetime(ventes_valides['date_vente'])

nb_jours_periode = (
    ventes_valides['date_vente'].max() - ventes_valides['date_vente'].min()
).days
nb_annees = max(nb_jours_periode / 365.25, 1)

qte_vendue = (
    ventes_valides.groupby('id_produit')['quantite']
    .sum()
    .reset_index()
    .rename(columns={'quantite': 'qte_vendue_totale'})
)
qte_vendue['ventes_annuelles_moy'] = qte_vendue['qte_vendue_totale'] / nb_annees

# =====================================================================
# ÉTAPE 3 : Fusionner et calculer la valeur du stock et la couverture en jours
# =====================================================================
df_surstock = (
    produits[['id_produit', 'nom_produit', 'cout_achat_unitaire']]
    .merge(stock_dispo, on='id_produit', how='left')
    .merge(
        qte_vendue[['id_produit', 'ventes_annuelles_moy']],
        on='id_produit',
        how='left',
    )
    .fillna(0)
)

# Valeur financière du stock immobilisé = quantité en stock * coût d'achat unitaire
df_surstock['valeur du stock'] = np.where(
    df_surstock['quantite en stock'] > 0,
    df_surstock['quantite en stock'] * df_surstock['cout_achat_unitaire'],
    0,
)

# Rotation annuelle puis couverture en jours (365 / rotation)
rotation = np.where(
    df_surstock['quantite en stock'] > 0,
    df_surstock['ventes_annuelles_moy'] / df_surstock['quantite en stock'],
    np.nan,
)
df_surstock['couverture_jour'] = np.where(rotation > 0, 365 / rotation, np.nan)

# =====================================================================
# ÉTAPE 4 : Filtrer les surstocks (> 90 jours) et afficher le total + Top 10
# =====================================================================
# On considère en surstock les produits avec un stock positif et une couverture > 90 jours
surstocks = df_surstock[
    (df_surstock['quantite en stock'] > 0)
    & (df_surstock['couverture_jour'] > 90)
].copy()

print(f'Nombre total de produits en surstock : {len(surstocks)}')

top10_surstock = (
    surstocks.sort_values(by='couverture_jour', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(
    top10_surstock[[
        'nom_produit',
        'quantite en stock',
        'valeur du stock',
        'couverture_jour',
    ]]
)

Nombre total de produits en surstock : 299


,nom_produit,quantite en stock,valeur du stock,couverture_jour
0,Eau Eco 357,2004.00,2627244.00,12239.53
1,Eau Premium 84,2211.00,1397352.00,8985.79
2,Savon Plus 105,1460.00,21593400.00,8917.02
3,Détergent Plus 18,1447.00,6395740.00,7989.57
4,Jus Classique 423,1350.00,22628700.00,7607.68
5,Farine Plus 68,1317.00,18477510.00,7577.96
6,Sucre Nature 314,1638.00,9688770.00,7430.48
7,Farine Plus 315,981.00,7679268.00,7055.79
8,Détergent Plus 12,1080.00,13828320.00,6945.36
9,Eau Classique 485,1141.00,518014.00,6891.71


Interprétation : Ces dix produits en surstock présentent une couverture très élevée (6 891,71 à 12 239,53 jours) et correspondent exactement aux références à faible rotation identifiées précédemment, mobilisant une valeur financière importante sur Jus Classique 423 (22 628 700,00 FCFA) et Savon Plus 105 (21 593 400,00 FCFA).

Conclusion métier : Geler les commandes sur ces références à faible rotation et mettre en place des actions commerciales ciblées pour écouler les stocks excédentaires et libérer de la trésorerie.


### 5.Fournisseurs les plus performants.

In [15]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: '%.2f' % x)

# =====================================================================
# ÉTAPE 1 : Convertir les dates de commande et de réception
# =====================================================================
# On travaille sur une copie de la table 'achats' pour ne pas modifier l'originale
df_achats = achats.copy()

# Conversion des dates en format datetime (errors='coerce' gère les dates invalides)
df_achats['date_commande'] = pd.to_datetime(
    df_achats['date_commande'], errors='coerce'
)
df_achats['date_reception'] = pd.to_datetime(
    df_achats['date_reception'], errors='coerce'
)

# =====================================================================
# ÉTAPE 2 : Calculer le délai de livraison (en jours) pour chaque commande
# =====================================================================
# On ne garde que les lignes où les deux dates sont présentes et valides
df_delais = df_achats.dropna(
    subset=['date_commande', 'date_reception']
).copy()

# Calcul de l'écart en jours entre la réception et la commande
df_delais['delai_jours'] = (
    df_delais['date_reception'] - df_delais['date_commande']
).dt.days

# On exclut les délais négatifs éventuels (anomalies de saisie)
df_delais = df_delais[df_delais['delai_jours'] >= 0]

# =====================================================================
# ÉTAPE 3 : Calculer le délai moyen de livraison par fournisseur
# =====================================================================
delai_fourn = (
    df_delais.groupby('id_fournisseur', as_index=False)['delai_jours']
    .mean()
    .rename(columns={'delai_jours': 'delai_moyen_livraison'})
)

# Fusion avec la table 'fournisseurs' pour récupérer le nom du fournisseur
df_fourn_perf = fournisseurs[['id_fournisseur', 'nom_fournisseur']].merge(
    delai_fourn, on='id_fournisseur', how='inner'
)

# =====================================================================
# ÉTAPE 4 : Trier par délai croissant et afficher le Top 10
# =====================================================================
# Un délai moyen plus faible indique un fournisseur plus rapide (plus performant)
top10_fournisseurs = (
    df_fourn_perf.sort_values(by='delai_moyen_livraison', ascending=True)
    .head(10)
    .reset_index(drop=True)
)

display(top10_fournisseurs[['nom_fournisseur', 'delai_moyen_livraison']])

,nom_fournisseur,delai_moyen_livraison
0,Fournisseur 093 SA,14.85
1,Fournisseur 050 SARL,15.90
2,Fournisseur 073 SARL,15.95
3,Fournisseur 096 SA,16.34
4,Fournisseur 016 SARL,16.40
5,Fournisseur 001 SARL,16.53
6,Fournisseur 020 SARL,16.54
7,Fournisseur 055 SARL,16.59
8,Fournisseur 021 SA,16.63
9,Fournisseur 049 SARL,16.73


### 6. Fournisseurs les moins performants.

In [16]:
import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: '%.2f' % x)

# =====================================================================
# ÉTAPE 1 : Convertir les dates de commande et de réception
# =====================================================================
# On travaille sur une copie de la table 'achats' pour ne pas modifier l'originale
df_achats = achats.copy()

# Conversion des dates en format datetime (errors='coerce' gère les dates invalides)
df_achats['date_commande'] = pd.to_datetime(
    df_achats['date_commande'], errors='coerce'
)
df_achats['date_reception'] = pd.to_datetime(
    df_achats['date_reception'], errors='coerce'
)

# =====================================================================
# ÉTAPE 2 : Calculer le délai de livraison (en jours) pour chaque commande
# =====================================================================
# On garde uniquement les lignes où les deux dates sont présentes et valides
df_delais = df_achats.dropna(
    subset=['date_commande', 'date_reception']
).copy()

# Calcul de l'écart en jours entre la réception et la commande
df_delais['delai_jours'] = (
    df_delais['date_reception'] - df_delais['date_commande']
).dt.days

# On exclut les délais négatifs éventuels (anomalies de saisie)
df_delais = df_delais[df_delais['delai_jours'] >= 0]

# =====================================================================
# ÉTAPE 3 : Calculer le délai moyen de livraison par fournisseur
# =====================================================================
delai_fourn = (
    df_delais.groupby('id_fournisseur', as_index=False)['delai_jours']
    .mean()
    .rename(columns={'delai_jours': 'delai_moyen_livraison'})
)

# Fusion avec la table 'fournisseurs' pour récupérer le nom du fournisseur
df_fourn_perf = fournisseurs[['id_fournisseur', 'nom_fournisseur']].merge(
    delai_fourn, on='id_fournisseur', how='inner'
)

# =====================================================================
# ÉTAPE 4 : Trier par délai décroissant et afficher le Flop 10
# =====================================================================
# Un délai moyen plus élevé indique un fournisseur plus lent (moins performant)
flop10_fournisseurs = (
    df_fourn_perf.sort_values(by='delai_moyen_livraison', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(flop10_fournisseurs[['nom_fournisseur', 'delai_moyen_livraison']])

,nom_fournisseur,delai_moyen_livraison
0,Fournisseur 072 SA,21.34
1,Fournisseur 071 SARL,20.90
2,Fournisseur 033 SA,20.60
3,Fournisseur 058 SARL,20.01
4,Fournisseur 034 SARL,19.93
5,Fournisseur 099 SA,19.75
6,Fournisseur 064 SARL,19.63
7,Fournisseur 048 SA,19.48
8,Fournisseur 069 SA,19.47
9,Fournisseur 023 SARL,19.19



•	Interprétation : Le délai moyen de livraison reste élevé sur l'ensemble des fournisseurs, variant de 14,85 à 16,73 jours pour le Top 10 (mené par Fournisseur 093 SA) et atteignant 19,19 à 21,34 jours pour le Flop 10 (Fournisseur 072 SA en tête).

•	Conclusion métier : Renégocier les délais contractuels des fournisseurs et privilégier les commandes auprès du Top 10 pour réduire le risque de rupture lié aux retards de livraison.


##  Detail.

#### 1. Etat des stocks actuels.

In [6]:


# 1. Détail par type de mouvement (Ajustement, Entrée, Retour, Sortie)
mouvements = stocks.pivot_table(
    index='id_produit',
    columns='type_mouvement',
    values='quantite',
    aggfunc='sum',
    fill_value=0
).reset_index()

# 2. Somme réelle de tous les mouvements (les sorties étant déjà négatives)
mouvements['stock_disponible'] = stocks.groupby('id_produit')['quantite'].sum().values

# Fusionner avec le catalogue produits
df_stock = pd.merge(produits[['id_produit', 'nom_produit', 'categorie']], mouvements, on='id_produit', how='left').fillna(0)

# Classification du statut (Rupture <= 0, Surstock > 500 unités)
df_stock['statut_stock'] = df_stock['stock_disponible'].apply(
    lambda x: 'Rupture' if x <= 0 else ('Surstock' if x > 500 else 'Normal')
)

print("Répartition globale des stocks :")
print(df_stock['statut_stock'].value_counts())

print("\nTop 10 des produits en RUPTURE (stock le plus bas) :")
display(df_stock.sort_values(by='stock_disponible', ascending=True)[['nom_produit', 'categorie', 'Ajustement', 'Entrée', 'Retour', 'Sortie', 'stock_disponible', 'statut_stock']].head(10))

print("\nTop 10 des produits en SURSTOCK (quantité la plus élevée) :")
display(df_stock.sort_values(by='stock_disponible', ascending=False)[['nom_produit', 'categorie', 'Ajustement', 'Entrée', 'Retour', 'Sortie', 'stock_disponible', 'statut_stock']].head(10))

Répartition globale des stocks :
statut_stock
Rupture     197
Normal      161
Surstock    143
Name: count, dtype: int64

Top 10 des produits en RUPTURE (stock le plus bas) :


,nom_produit,categorie,Ajustement,Entrée,Retour,Sortie,stock_disponible,statut_stock
234,Biscuit Plus 235,Produits Alimentaires,199.00,906.00,0.00,-2653.00,-1548.00,Rupture
401,Jus Nature 402,Boisson,56.00,1589.00,0.00,-3126.00,-1481.00,Rupture
489,Sucre Nature 490,Épicerie,178.00,796.00,18.00,-2160.00,-1168.00,Rupture
54,Café Plus 55,Boisson,72.00,649.00,0.00,-1865.00,-1144.00,Rupture
78,Huile Famille 79,Épicerie,181.00,781.00,0.00,-2064.00,-1102.00,Rupture
247,Savon Eco 248,Entretien,287.00,349.00,129.00,-1821.00,-1056.00,Rupture
70,Jus Famille 71,Boisson,336.00,780.00,23.00,-2192.00,-1053.00,Rupture
46,Eau Famille 47,Boisson,112.00,811.00,88.00,-2057.00,-1046.00,Rupture
390,Café Plus 391,Boisson,275.00,760.00,35.00,-2069.00,-999.00,Rupture
47,Sucre Nature 48,Épicerie,234.00,998.00,0.00,-2231.00,-999.00,Rupture



Top 10 des produits en SURSTOCK (quantité la plus élevée) :


,nom_produit,categorie,Ajustement,Entrée,Retour,Sortie,stock_disponible,statut_stock
83,Eau Premium 84,Boisson,459.00,2231.00,838.00,-1317.00,2211.00,Surstock
356,Eau Eco 357,Boisson,272.00,2486.00,141.00,-895.00,2004.00,Surstock
313,Sucre Nature 314,Épicerie,276.00,1915.00,461.00,-1014.00,1638.00,Surstock
243,Sucre Premium 244,Épicerie,367.00,2291.00,408.00,-1524.00,1542.00,Surstock
45,Détergent Classique 46,Entretien,155.00,1776.00,400.00,-790.00,1541.00,Surstock
104,Savon Plus 105,Entretien,398.00,1743.00,149.00,-830.00,1460.00,Surstock
17,Détergent Plus 18,Entretien,247.00,2333.00,153.00,-1286.00,1447.00,Surstock
422,Jus Classique 423,Boisson,543.00,1659.00,267.00,-1119.00,1350.00,Surstock
67,Farine Plus 68,Épicerie,422.00,1668.00,354.00,-1127.00,1317.00,Surstock
332,Lait Famille 333,Boisson,434.00,1496.00,376.00,-1016.00,1290.00,Surstock


Explication métier des KPI calculés
Mouvements nets par type (Entrée, Sortie, Ajustement, Retour) : Mesurent l'ensemble des flux physiques entrants (réceptions fournisseurs, retours clients, régularisations positives) et sortants (ventes, casses, transferts).

Stock disponible (stock_disponible) : Quantité nette théorique présente en entrepôt (Entrées + Ajustements + Retours - Sorties). C'est le niveau réel disponible pour honorer les commandes clients.

Statut du stock (Rupture, Normal, Surstock) : Indicateur d'alerte opérationnelle permettant de segmenter le catalogue entre les produits indisponibles (<=0>), équilibrés (1 à 500 unités) et excessifs (> 500 unités).


Interprétation des résultats
Répartition déséquilibrée : Sur 501 références, 197 sont en rupture (39,3 %), 143 en surstock (28,5 %) et seulement 161 en niveau normal (32,1 %).

Anomalie de flux sur les ruptures : Les 10 pires ruptures affichent des soldes fortement négatifs (jusqu'à -1 548 unités sur Biscuit Plus 235). Cela révèle l'absence d'enregistrement du stock initial dans la base et/ou des oublis de saisie lors des réceptions de marchandises.

Accumulation sur les surstocks : À l'inverse, des produits comme Eau Premium 84 (+2 211 unités) ou Eau Eco 357 (+2 004 unités) accumulent des volumes massifs en raison d'achats non alignés sur le rythme réel des ventes.


Conclusion métier
67,8 % du catalogue est en situation critique (rupture ou surstock), traduisant un pilotage des approvisionnements déconnecté de la demande et des failles dans la traçabilité des entrées en entrepôt. Un inventaire physique général s'impose pour remettre les compteurs informatiques à zéro, accompagné d'un arrêt immédiat des commandes sur les 143 références en surstock.

#### 2. Performance des stocks.

In [7]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 1. Stock disponible par produit
stock_prod = stocks.groupby('id_produit')['quantite'].sum().reset_index().rename(columns={'quantite': 'stock_disponible'})

# 2. Ventes annuelles moyennes (en quantité) sur les ventes validées
ventes_valides = ventes[ventes['statut_vente'].str.lower().str.contains('valid', na=False)].copy()
ventes_valides['date_vente'] = pd.to_datetime(ventes_valides['date_vente'])
nb_jours_periode = (ventes_valides['date_vente'].max() - ventes_valides['date_vente'].min()).days
nb_annees = max(nb_jours_periode / 365.25, 1)

qte_vendue = ventes_valides.groupby('id_produit')['quantite'].sum().reset_index().rename(columns={'quantite': 'qte_vendue_totale'})
qte_vendue['ventes_annuelles_moy'] = qte_vendue['qte_vendue_totale'] / nb_annees

# 3. Fusion avec le catalogue produits
df_perf = produits[['id_produit', 'nom_produit', 'categorie', 'cout_achat_unitaire']].merge(stock_prod, on='id_produit', how='left').merge(qte_vendue, on='id_produit', how='left').fillna(0)

# 4. Calcul des indicateurs de performance (sur les produits ayant un stock positif)
df_perf['valeur_stock_fcfa'] = np.where(df_perf['stock_disponible'] > 0, df_perf['stock_disponible'] * df_perf['cout_achat_unitaire'], 0)
df_perf['rotation_stock'] = np.where(df_perf['stock_disponible'] > 0, df_perf['ventes_annuelles_moy'] / df_perf['stock_disponible'], np.nan)
df_perf['couverture_jours'] = np.where(df_perf['rotation_stock'] > 0, 365 / df_perf['rotation_stock'], np.nan)

# Nouvelle définition métier du surstock (> 90 jours de couverture)
df_perf['diagnostic_reel'] = np.where(
    df_perf['stock_disponible'] <= 0, 'Rupture',
    np.where(df_perf['couverture_jours'] > 90, 'Surstock (>90j)', 'Stock sain')
)

print(f"Valeur financière totale du stock positif immobilisé : {df_perf['valeur_stock_fcfa'].sum():,.0f} FCFA")
print("\nRépartition selon la couverture réelle en jours :")
print(df_perf['diagnostic_reel'].value_counts())

print("\nTop 10 des produits immobilisant le plus de trésorerie (Valeur financière) :")
display(df_perf.sort_values(by='valeur_stock_fcfa', ascending=False)[['nom_produit', 'categorie', 'stock_disponible', 'valeur_stock_fcfa', 'rotation_stock', 'couverture_jours', 'diagnostic_reel']].head(10))

print("\nTop 10 des produits à plus faible rotation (Durée de stockage la plus longue) :")
display(df_perf[df_perf['stock_disponible'] > 0].sort_values(by='couverture_jours', ascending=False)[['nom_produit', 'categorie', 'stock_disponible', 'ventes_annuelles_moy', 'rotation_stock', 'couverture_jours']].head(10))

Valeur financière totale du stock positif immobilisé : 1,915,323,501 FCFA

Répartition selon la couverture réelle en jours :
diagnostic_reel
Surstock (>90j)    299
Rupture            197
Stock sain           5
Name: count, dtype: int64

Top 10 des produits immobilisant le plus de trésorerie (Valeur financière) :


,nom_produit,categorie,stock_disponible,valeur_stock_fcfa,rotation_stock,couverture_jours,diagnostic_reel
304,Savon Nature 305,Entretien,1203.00,29515605.00,0.08,4663.76,Surstock (>90j)
0,Jus Classique 1,Boisson,1231.00,28529656.00,0.05,6695.48,Surstock (>90j)
24,Biscuit Premium 25,Produits Alimentaires,1119.00,25025316.00,0.06,5689.99,Surstock (>90j)
332,Lait Famille 333,Boisson,1290.00,24729300.00,0.07,5552.34,Surstock (>90j)
179,Jus Plus 180,Boisson,1118.00,23465702.00,0.06,6399.24,Surstock (>90j)
389,Jus Nature 390,Boisson,1025.00,23181400.00,0.08,4377.28,Surstock (>90j)
69,Sucre Plus 70,Épicerie,1275.00,22697550.00,0.06,6423.48,Surstock (>90j)
422,Jus Classique 423,Boisson,1350.00,22628700.00,0.05,7607.68,Surstock (>90j)
104,Savon Plus 105,Entretien,1460.00,21593400.00,0.04,8917.02,Surstock (>90j)
363,Pâtes Nature 364,Entretien,1113.00,21221571.00,0.06,5794.23,Surstock (>90j)



Top 10 des produits à plus faible rotation (Durée de stockage la plus longue) :


,nom_produit,categorie,stock_disponible,ventes_annuelles_moy,rotation_stock,couverture_jours
356,Eau Eco 357,Boisson,2004.00,59.76,0.03,12239.53
83,Eau Premium 84,Boisson,2211.00,89.81,0.04,8985.79
104,Savon Plus 105,Entretien,1460.00,59.76,0.04,8917.02
17,Détergent Plus 18,Entretien,1447.00,66.11,0.05,7989.57
422,Jus Classique 423,Boisson,1350.00,64.77,0.05,7607.68
67,Farine Plus 68,Épicerie,1317.00,63.43,0.05,7577.96
313,Sucre Nature 314,Épicerie,1638.00,80.46,0.05,7430.48
314,Farine Plus 315,Épicerie,981.00,50.75,0.05,7055.79
11,Détergent Plus 12,Entretien,1080.00,56.76,0.05,6945.36
484,Eau Classique 485,Boisson,1141.00,60.43,0.05,6891.71


Explication des KPI
Valeur du stock immobilisé (valeur_stock_fcfa) : Capital financier bloqué dans l'entrepôt (Stock disponible * Coût d'achat unitaire).

Taux de rotation (rotation_stock) : Nombre de fois où le stock est renouvelé sur un an (Ventes annuelles moyennes / Stock disponible). Plus il est élevé, plus le produit s'écoule vite.

Couverture en jours (couverture_jours) : Nombre de jours de ventes assurés par le stock actuel (365 / Rotation). Au-delà de 90 jours, le produit est classé en surstock réel.


Interprétation
Trésorerie paralysée : 1,91 milliard FCFA sont immobilisés dans les stocks positifs, dont 29,5 M FCFA sur le seul Savon Nature 305.

Surstockage quasi généralisé : 299 produits (59,7 %) dépassent les 90 jours de couverture, contre seulement 5 produits (1 %) ayant un stock sain.

Rotation quasi nulle : Les produits dorment des années en entrepôt (rotation entre 0,03 et 0,08), comme Eau Eco 357 qui représente 12 239 jours (~33 ans) de ventes en stock.


Conclusion métier
L'entreprise immobilise près de 2 milliards FCFA dans des stocks à très faible rotation tout en subissant des ruptures sur près de 40 % de son catalogue. Il est urgent de liquider les surstocks (promotions, déstockage) pour libérer du cash et réallouer cette trésorerie vers le réapprovisionnement des produits en rupture.

#### 3. Fournisseurs & repprovisionnement.

In [8]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 1. Calcul des délais réels d'approvisionnement (sur les commandes reçues)
achats['date_commande'] = pd.to_datetime(achats['date_commande'])
achats['date_reception'] = pd.to_datetime(achats['date_reception'])
achats_recus = achats.dropna(subset=['date_reception']).copy()
achats_recus['delai_reel_jours'] = (achats_recus['date_reception'] - achats_recus['date_commande']).dt.days

# Fusion avec la table fournisseurs pour comparer au délai contractuel
df_fourn = achats_recus.merge(fournisseurs[['id_fournisseur', 'nom_fournisseur', 'type_fournisseur', 'delai_contractuel_jours']], on='id_fournisseur', how='left')
df_fourn['retard_jours'] = df_fourn['delai_reel_jours'] - df_fourn['delai_contractuel_jours']
df_fourn['en_retard'] = np.where(df_fourn['retard_jours'] > 0, 1, 0)

delai_moyen_global = df_fourn['delai_reel_jours'].mean()
taux_retard_global = df_fourn['en_retard'].mean() * 100

print(f"Délai moyen global d'approvisionnement : {delai_moyen_global:.1f} jours")
print(f"Taux global de commandes livrées en retard : {taux_retard_global:.1f} %")

# 2. Performance des fournisseurs (Top 10 pires retards moyens, avec au moins 10 commandes)
perf_fournisseurs = df_fourn.groupby(['id_fournisseur', 'nom_fournisseur', 'type_fournisseur']).agg(
    nb_commandes=('id_achat', 'count'),
    delai_contractuel=('delai_contractuel_jours', 'mean'),
    delai_reel_moy=('delai_reel_jours', 'mean'),
    retard_moyen=('retard_jours', 'mean'),
    taux_retard_pct=('en_retard', lambda x: x.mean() * 100)
).reset_index()

print("\nTop 10 des fournisseurs les moins performants (plus fort retard moyen) :")
display(perf_fournisseurs[perf_fournisseurs['nb_commandes'] >= 10].sort_values(by='retard_moyen', ascending=False).head(10))

# 3. Calcul du Stock Minimum, Point de Commande et Top 20 produits problématiques (sur df_perf de l'étape 2)
delai_par_produit = df_fourn.groupby('id_produit')['delai_reel_jours'].mean().reset_index().rename(columns={'delai_reel_jours': 'delai_appro_moy'})
df_opti = df_perf.merge(delai_par_produit, on='id_produit', how='left')
df_opti['delai_appro_moy'] = df_opti['delai_appro_moy'].fillna(delai_moyen_global)

# Ventes journalières moyennes
df_opti['ventes_jour_moy'] = df_opti['ventes_annuelles_moy'] / 365.0

# Stock minimum (sécurité = 15 jours de ventes) et Seuil de réapprovisionnement (Point de commande)
df_opti['stock_minimum_securite'] = np.ceil(df_opti['ventes_jour_moy'] * 15)
df_opti['seuil_reapprovisionnement'] = np.ceil((df_opti['ventes_jour_moy'] * df_opti['delai_appro_moy']) + df_opti['stock_minimum_securite'])

# Top 20 des produits les plus problématiques (10 pires ruptures + 10 plus gros surstocks financiers)
top10_ruptures = df_opti.sort_values(by='stock_disponible', ascending=True).head(10).copy()
top10_ruptures['probleme'] = 'Rupture critique'

top10_surstocks = df_opti.sort_values(by='valeur_stock_fcfa', ascending=False).head(10).copy()
top10_surstocks['probleme'] = 'Surstock financier majeur'

top20_problematiques = pd.concat([top10_ruptures, top10_surstocks], ignore_index=True)

print("\nTOP 20 DES PRODUITS LES PLUS PROBLÉMATIQUES & SEUILS RECOMMANDÉS :")
display(top20_problematiques[['probleme', 'nom_produit', 'categorie', 'stock_disponible', 'valeur_stock_fcfa', 'delai_appro_moy', 'stock_minimum_securite', 'seuil_reapprovisionnement']])

Délai moyen global d'approvisionnement : 18.0 jours
Taux global de commandes livrées en retard : 51.0 %

Top 10 des fournisseurs les moins performants (plus fort retard moyen) :


,id_fournisseur,nom_fournisseur,type_fournisseur,nb_commandes,delai_contractuel,delai_reel_moy,retard_moyen,taux_retard_pct
63,FOU064,Fournisseur 064 SARL,Grossiste,102,2.00,19.63,17.63,92.16
22,FOU023,Fournisseur 023 SARL,Grossiste,94,2.00,19.19,17.19,96.81
50,FOU051,Fournisseur 051 SA,International,94,2.00,18.84,16.84,96.81
21,FOU022,Fournisseur 022 SARL,International,96,4.00,18.52,14.52,90.62
87,FOU088,Fournisseur 088 SARL,Grossiste,97,3.00,17.38,14.38,88.66
99,FOU100,Fournisseur 100 SARL,Local,84,3.00,17.12,14.12,88.10
42,FOU043,Fournisseur 043 SARL,Grossiste,112,4.00,17.94,13.94,90.18
40,FOU041,Fournisseur 041 SARL,International,83,5.00,18.67,13.67,85.54
1,FOU002,Fournisseur 002 SARL,Local,95,5.00,18.62,13.62,88.42
59,FOU060,Fournisseur 060 SA,International,90,5.00,17.64,12.64,84.44



TOP 20 DES PRODUITS LES PLUS PROBLÉMATIQUES & SEUILS RECOMMANDÉS :


,probleme,nom_produit,categorie,stock_disponible,valeur_stock_fcfa,delai_appro_moy,stock_minimum_securite,seuil_reapprovisionnement
0,Rupture critique,Biscuit Plus 235,Produits Alimentaires,-1548.00,0.00,22.93,5.00,12.00
1,Rupture critique,Jus Nature 402,Boisson,-1481.00,0.00,15.48,4.00,8.00
2,Rupture critique,Sucre Nature 490,Épicerie,-1168.00,0.00,16.44,3.00,7.00
3,Rupture critique,Café Plus 55,Boisson,-1144.00,0.00,15.69,3.00,6.00
4,Rupture critique,Huile Famille 79,Épicerie,-1102.00,0.00,20.27,3.00,7.00
5,Rupture critique,Savon Eco 248,Entretien,-1056.00,0.00,18.83,3.00,7.00
6,Rupture critique,Jus Famille 71,Boisson,-1053.00,0.00,22.28,5.00,12.00
7,Rupture critique,Eau Famille 47,Boisson,-1046.00,0.00,18.87,4.00,8.00
8,Rupture critique,Café Plus 391,Boisson,-999.00,0.00,19.67,4.00,10.00
9,Rupture critique,Sucre Nature 48,Épicerie,-999.00,0.00,19.27,5.00,11.00


Explication des KPI
Délai réel d'approvisionnement (delai_reel_jours) : Temps écoulé entre la commande et la réception physique (Date réception - Date commande).

Taux et durée de retard (retard_jours, taux_retard_pct) : Écart entre le délai réel et le délai contractuel promis par le fournisseur, et pourcentage de livraisons hors délai.

Stock minimum de sécurité (stock_minimum_securite) : Réserve tampon (15 jours de ventes) pour absorber les retards fournisseurs ou pics de demande.

Point de commande (seuil_reapprovisionnement) : Niveau de stock déclenchant automatiquement une nouvelle commande (Ventes pendant le délai fournisseur + Stock minimum).


Interprétation
Fournisseurs peu fiables : Le délai moyen est de 18,0 jours, avec 51,0 % des commandes livrées en retard. Les pires fournisseurs (Fournisseur 064 SARL, Fournisseur 023 SARL, Fournisseur 051 SA) affichent plus de 92 % à 96 % de retards, livrant en ~19 jours au lieu des 2 jours contractuels.

Déconnexion des quantités commandées : Les seuils de réapprovisionnement optimaux (point de commande) se situent entre 6 et 12 unités selon la demande réelle, alors que l'entreprise commande et stocke plus de 1 000 à 1 400 unités sur des produits comme Savon Nature 305 ou Jus Classique 1.


Conclusion métier
La politique d'achat actuelle subit à la fois des retards fournisseurs chroniques (1 commande sur 2) et une absence de règles de commande (achats massifs au lieu de suivre la demande réelle). Il faut renégocier les contrats avec les fournisseurs défaillants (pénalités de retard, diversification) et automatiser les commandes sur la base des seuils de réapprovisionnement calculés (entre 6 et 12 unités).